## 6. Relational Merge: Creating the Unified District Master Profile (2026)
Consolidating District Footfall (2026), NIDHI Hospitality Capacity, and ASI Heritage Monuments into a single unified analytical dataset.


In [ ]:
import pandas as pd
import numpy as np

# Load cleaned datasets with engineered features
df_district_footfall = pd.read_csv('../data/cleaned/01_district_wise_tourist_footfall_cleaned.csv')
df_nidhi_hospitality = pd.read_csv('../data/cleaned/02_nidhi_hospitality_establishments_cleaned.csv')
df_asi_monuments = pd.read_csv('../data/cleaned/03_asi_protected_monuments_cleaned.csv')
print('Datasets loaded for Relational Merge.')


In [127]:
# 1. Aggregate NIDHI Hospitality by District
nidhi_district_agg = df_nidhi_hospitality.groupby(['State_UT', 'District_City']).agg(
    Nidhi_Total_Properties=('Establishment_ID', 'count'),
    Nidhi_Total_Rooms=('Total_Rooms', 'sum'),
    Nidhi_Avg_Min_Tariff_INR=('Room_Tariff_Min_INR', 'mean'),
    Nidhi_Star_Hotels_Count=('Star_Classification', lambda s: s.astype(str).str.contains('Star').sum())
).reset_index().rename(columns={'District_City': 'District'})

# 2. Aggregate ASI Monuments by District
asi_district_agg = df_asi_monuments.groupby(['State_UT', 'District']).agg(
    ASI_Total_Monuments=('Monument_ID', 'count'),
    ASI_UNESCO_Sites=('UNESCO_Heritage_Status', lambda s: (s == 'World Heritage Site').sum()),
    ASI_Total_Footfall_2026=('Estimated_Annual_Footfall_2026', 'sum')
).reset_index()

# 3. Left-join with 2026 District Footfall
df_district_2026 = df_district_footfall[df_district_footfall['Year'] == 2026].copy()
df_district_master = (
    df_district_2026
    .merge(nidhi_district_agg, on=['State_UT', 'District'], how='left')
    .merge(asi_district_agg, on=['State_UT', 'District'], how='left')
)

# Fill non-mapped counts with 0
df_district_master['Nidhi_Total_Properties'] = df_district_master['Nidhi_Total_Properties'].fillna(0).astype('Int64')
df_district_master['Nidhi_Total_Rooms'] = df_district_master['Nidhi_Total_Rooms'].fillna(0).astype('Int64')
df_district_master['Nidhi_Star_Hotels_Count'] = df_district_master['Nidhi_Star_Hotels_Count'].fillna(0).astype('Int64')
df_district_master['ASI_Total_Monuments'] = df_district_master['ASI_Total_Monuments'].fillna(0).astype('Int64')
df_district_master['ASI_UNESCO_Sites'] = df_district_master['ASI_UNESCO_Sites'].fillna(0).astype('Int64')

# Compute Infrastructure Stress Ratio: Visitors per available registered hotel room
df_district_master['Visitors_Per_Hotel_Room'] = np.where(
    df_district_master['Nidhi_Total_Rooms'] > 0,
    (df_district_master['Total_Visitors'] / df_district_master['Nidhi_Total_Rooms']).round(1),
    np.nan
)

print(f'df_district_master created! Total 2026 Districts: {len(df_district_master)}')
# Verify with .head()
display(df_district_master[['District', 'State_UT', 'Total_Visitors', 'Foreign_Visitors', 'Nidhi_Total_Rooms', 'ASI_Total_Monuments', 'Visitors_Per_Hotel_Room']].head(5))


df_district_master created! Total 2026 Districts: 476


,District,State_UT,Total_Visitors,Foreign_Visitors,Nidhi_Total_Rooms,ASI_Total_Monuments,Visitors_Per_Hotel_Room
0,Agra,Uttar Pradesh,2739650,63106,0,1,NaN
1,Varanasi,Uttar Pradesh,4587441,158230,12,2,382286.8
2,Lucknow,Uttar Pradesh,363342,2047,19,1,19123.3
3,Prayagraj,Uttar Pradesh,784003,6849,13,0,60307.9
4,Ayodhya,Uttar Pradesh,739680,438,0,2,NaN
